In [41]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-a8kdkzku
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-a8kdkzku
  Resolved https://github.com/huggingface/transformers.git to commit f15fc1e8f8939cdd1d4e54c5cd409317c5a9e5ce
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [42]:
!pip install -U accelerate torch torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
data_dir = "/content/drive/MyDrive/data/"

In [45]:
from __future__ import annotations

import json
import re
from decimal import Decimal
from pathlib import Path
from typing import Optional

from pydantic import BaseModel, Field


In [60]:
# ---------------------------------------------------------------------------
# Pydantic models
# ---------------------------------------------------------------------------

from pydantic import model_validator

class LineItem(BaseModel):
    """A single purchased item on the receipt."""
    item: str
    qty: int = 1
    unit_price: Optional[float] = None
    total_price: Optional[float] = None


class Financials(BaseModel):
    """Aggregate monetary fields printed on the receipt."""
    subtotal: Optional[float] = None
    tax: Optional[float] = 0.0
    tip: Optional[float] = 0.0
    grand_total: float

    @model_validator(mode='before')
    @classmethod
    def coerce_none_to_zero(cls, values):
        """Replace None with 0.0 for tax/tip so downstream math works."""
        for field in ('tax', 'tip'):
            if values.get(field) is None:
                values[field] = 0.0
        return values


class ReceiptData(BaseModel):
    """Full structured payload extracted from a receipt image."""
    raw_transcription: str
    total_items_counted: int
    merchant_name: Optional[str] = None
    date: Optional[str] = None
    line_items: list[LineItem] = Field(default_factory=list)
    financials: Financials


class ValidationResult(BaseModel):
    """Result of comparing the computed total to the printed total."""
    is_match: bool
    calculated_total: float
    printed_total: float
    difference: float


In [47]:
# ---------------------------------------------------------------------------
# Custom exception
# ---------------------------------------------------------------------------

class RequiresHumanInterventionError(Exception):
    """Raised when automated extraction cannot reconcile totals."""

    def __init__(
        self,
        message: str,
        receipt_data: ReceiptData,
        validation: ValidationResult,
    ) -> None:
        super().__init__(message)
        self.receipt_data = receipt_data
        self.validation = validation

In [48]:
# ---------------------------------------------------------------------------
# Deterministic arithmetic tool (bound to the LLM pipeline)
# ---------------------------------------------------------------------------

def calculate_actual_total(line_items: list[float]) -> float:
    """Sum line-item prices using Decimal to avoid float rounding errors.

    This is the *tool* the system calls instead of letting the LLM
    hallucinate arithmetic.
    """
    total = sum(Decimal(str(p)) for p in line_items)
    return float(total)


In [49]:
EXTRACTION_PROMPT = """\
Analyze the receipt image and return ONLY a JSON object (no markdown fences).
IMPORTANT — follow these steps IN ORDER to avoid skipping line items:

1. Transcribe: Read every line of text on the receipt from top to bottom,
    exactly as printed, and write it into the "raw_transcription" field.
    Include item names, prices, subtotals, tax lines — everything.
2. Count: Count the number of purchased line items (not subtotal/tax/total
    lines) and write it into "total_items_counted".
3. Structure: Using ONLY the transcription above, fill in the
    "line_items" and "financials" fields.  The number of objects in
    "line_items" MUST equal "total_items_counted".

Doing the transcription and count first prevents you from accidentally skipping any items.
  
{
  "merchant_name": "",
  "date": "",
  "line_items": [
    {"item": "", "qty": 0, "unit_price": 0.00, "total_price": 0.00}
  ],
  "financials": {
    "subtotal": 0.00,
    "tax": 0.00,
    "tip": 0.00,
    "grand_total": 0.00
  }
}

Rules:
- Extract every purchased line item with its name, quantity, unit price, and
  total price.
- Pay close attention to quantities. If a line shows "2 @", "x6", "QTY 6",
  or a multiplier before/after the item name, set "qty" to that number and
  "unit_price" to the per-unit price.  "total_price" = qty * unit_price. 
  Do NOT put the line total in "unit_price" — "unit_price" is always the per-single-item price.
- If an item appears with no quantity indicator, default qty to 1.
- Identify the printed grand total from the footer.
- Use null for any field you cannot read.
- Do NOT guess arithmetic; just transcribe what is printed.
"""


In [50]:
def _build_rescan_prompt(base_prompt: str, difference: float) -> str:
    """Append critic feedback for the retry attempt."""
    abs_diff = abs(difference)
    if difference < 0:
        # Extracted items sum to MORE than grand total
        guidance = (
        f"Your extracted line-item total is ${abs_diff:.2f} HIGHER than the "
        f"printed grand total. You likely hallucinated an extra item or "
        f"read a price too high. Look for a line item that should be "
        f"removed or whose price should be reduced by ${abs_diff:.2f}."
        )
    else:
        # Extracted items sum to LESS than grand total
        guidance = (
        f"Your extracted line-item total is ${abs_diff:.2f} LOWER than the "
        f"printed grand total. You likely missed a line item. Look "
        f"specifically for an item or a combination of items that cost "
        f"exactly ${abs_diff:.2f}."
    )
    return (
        base_prompt
        + f"\\n\\nCRITIC FEEDBACK: The previous extraction had a discrepancy "
        + f"of ${abs_diff:.2f}. {guidance} "
        + f"Please re-examine the receipt carefully, redo the full "
        + f"transcription, recount, and correct the structured output."
    )

In [51]:
def ask_vlm(
    image_path: str,
    prompt: str,
    processor: object,
    model: object,
) -> Optional[dict]:
    """Send an image + prompt to the VLM and parse the JSON response.

    Uses the same calling convention as the reference notebook:
    processor.apply_chat_template → model.generate → processor.decode.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]
    outputs = model.generate(**inputs, max_new_tokens=2048)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

    # Gemma-family models expose parse_response; fall back to raw text.
    if hasattr(processor, "parse_response"):
        text = processor.parse_response(response).get("content", response)
    else:
        text = response

    # Extract the first JSON object from the response.
    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            return None
    return None


In [52]:
# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

def _validate(receipt: ReceiptData) -> ValidationResult:
    """Compare deterministic sum of line items against the printed total."""
    prices = [item.total_price for item in receipt.line_items]
    computed = Decimal(str(calculate_actual_total(prices)))
    computed += Decimal(str(receipt.financials.tax))
    computed += Decimal(str(receipt.financials.tip))

    printed = Decimal(str(receipt.financials.grand_total))
    diff = float(computed - printed)

    return ValidationResult(
        is_match=abs(diff) < 0.01,
        calculated_total=float(computed),
        printed_total=float(printed),
        difference=diff,
    )



In [53]:
# ---------------------------------------------------------------------------
# Core pipeline
# ---------------------------------------------------------------------------

MAX_RETRIES = 1


def process_receipt(image_path: str, processor: object | None = None, model: object | None = None) -> ReceiptData:
    """End-to-end receipt processing with closed-loop validation.

    Args:
        image_path: Path (or URL) to the receipt image.
        processor: A loaded HF processor (e.g. AutoProcessor).
        model: A loaded HF VLM (e.g. AutoModelForMultimodalLM).

    Returns:
        A validated ReceiptData object.

    Raises:
        RequiresHumanInterventionError: If the totals cannot be reconciled
            after one retry.
        FileNotFoundError: If the image path does not exist.
        ValueError: If the VLM fails to return parseable JSON on both attempts.
    """
    # Validate input path exists (skip check for URLs).
    if not image_path.startswith(("http://", "https://")):
        if not Path(image_path).exists():
            raise FileNotFoundError(f"Receipt image not found: {image_path}")

    # Load model lazily if none provided.
    if processor is None or model is None:
        from transformers import AutoModelForMultimodalLM, AutoProcessor

        model_id = "google/gemma-4-E2B-it"
        processor = AutoProcessor.from_pretrained(model_id)
        model = AutoModelForMultimodalLM.from_pretrained(
            model_id, dtype="auto", device_map="auto",
        )

    prompt = EXTRACTION_PROMPT
    receipt: Optional[ReceiptData] = None
    validation: Optional[ValidationResult] = None

    for attempt in range(MAX_RETRIES + 1):
        # --- Step 1: VLM extraction ---
        raw = ask_vlm(image_path, prompt, processor, model)
        if raw is None:
            if attempt < MAX_RETRIES:
                prompt = _build_rescan_prompt(EXTRACTION_PROMPT, 0.0)
                continue
            raise ValueError(
                "VLM failed to return parseable JSON after all attempts."
            )

        raw["raw_transcription"] = json.dumps(raw)
        raw["total_items_counted"] = len(raw.get("line_items", []))
        
        receipt = ReceiptData.model_validate(raw)

        # --- Step 2: Deterministic arithmetic via tool ---
        validation = _validate(receipt)

        # --- Step 3: Check match ---
        if validation.is_match:
            return receipt

        # --- Step 4: Prepare retry with critic feedback ---
        if attempt < MAX_RETRIES:
            prompt = _build_rescan_prompt(EXTRACTION_PROMPT, validation.difference)

    # --- Step 5: Human intervention fallback ---
    assert receipt is not None and validation is not None
    raise RequiresHumanInterventionError(
        message=(
            f"Discrepancy of {validation.difference:.2f} persists after "
            f"{MAX_RETRIES + 1} attempts. Requires manual review."
        ),
        receipt_data=receipt,
        validation=validation,
    )


In [54]:
GEMMA_ID = "google/gemma-4-31B-it"
LLAMA_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"
QWEN_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

In [55]:
from huggingface_hub import notebook_login
notebook_login()

In [66]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(QWEN_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto"
)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [ ]:
# from transformers import AutoProcessor, AutoModelForMultimodalLM

# # Load model
# processor = AutoProcessor.from_pretrained(GEMMA_ID, visual_token_budget=1120)
# model = AutoModelForMultimodalLM.from_pretrained(
#     GEMMA_ID,
#     dtype="auto",
#     device_map="auto"
# )

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

In [ ]:
# from transformers import MllamaForConditionalGeneration, AutoProcessor

# model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
# processor = AutoProcessor.from_pretrained(LLAMA_ID)
# model = MllamaForConditionalGeneration.from_pretrained(
#     model_id, torch_dtype="auto", device_map="auto",
# )


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct.
403 Client Error. (Request ID: Root=1-6a020bf5-6389d16e673496577828d2c1;e13e8880-0b2c-4620-af1e-5562811193b6)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-11B-Vision-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct to ask for access.

In [69]:
# --- Execution ---
import sys
import os
#image_filename = "Receipt - Carnegie Mellon University Store - July 10, 2025.jpg"
#image_filename = "Receipt - Metro City Restaurant & Bar - Jul 6, 2025.jpg"
#image_filename = "Receipt - Trader Joe_s - July, 7, 2025.jpg"
#image_filename = "Receipt - Costco Wholesale - Jul 17, 2025.jpg"
image_filename = "Receipt - Target - Jul 13, 2025.jpg"
image_path = os.path.join(data_dir, image_filename)
try:
    result = process_receipt(image_path)
    print(result.model_dump_json(indent=2))
except RequiresHumanInterventionError as exc:
    print(f"⚠ HUMAN REVIEW REQUIRED: {exc}")
    print("Extracted so far:")
    print(exc.receipt_data.model_dump_json(indent=2))
    print("Validation:")
    print(exc.validation.model_dump_json(indent=2))
    sys.exit(2)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



⚠ HUMAN REVIEW REQUIRED: Discrepancy of 10.10 persists after 2 attempts. Requires manual review.
Extracted so far:
{
  "raw_transcription": "{\"merchant_name\": \"East Liberty\", \"date\": \"07/13/2025\", \"line_items\": [{\"item\": \"OLIPOP Regular Price\", \"qty\": 1, \"unit_price\": 9.29, \"total_price\": 9.29}, {\"item\": \"BOG010% Circle\", \"qty\": 1, \"unit_price\": 8.83, \"total_price\": 8.83}, {\"item\": \"SIMPLY Regular Price\", \"qty\": 1, \"unit_price\": 9.29, \"total_price\": 9.29}, {\"item\": \"BOG010% Circle\", \"qty\": 1, \"unit_price\": 8.83, \"total_price\": 8.83}, {\"item\": \"FD ICE CREAM\", \"qty\": 1, \"unit_price\": 2.99, \"total_price\": 2.99}, {\"item\": \"SIMPLY\", \"qty\": 1, \"unit_price\": 2.69, \"total_price\": 2.69}, {\"item\": \"GHOST ENERGY\", \"qty\": 1, \"unit_price\": 2.69, \"total_price\": 2.69}], \"financials\": {\"subtotal\": 34.51, \"tax\": 7.0, \"tip\": 0.0, \"grand_total\": 41.51}}",
  "total_items_counted": 7,
  "merchant_name": "East Liberty"

TypeError: object of type 'NoneType' has no len()